In [1]:
import boto3
import pandas as pd

s3 = boto3.client("s3")

bucket_name = "dsan6000-bm1262"
prefix = "wikipedia-hourly/"

In [2]:
response = s3.list_objects_v2(
    Bucket=bucket_name,
    Prefix=prefix
)

s3_keys = [
    obj["Key"]
    for obj in response.get("Contents", [])
    if obj["Key"].endswith(".parquet")
]

print(f"Number of files: {len(s3_keys)}")

Number of files: 24


In [3]:
dfs = []

for key in sorted(s3_keys):
    s3_uri = f"s3://{bucket_name}/{key}"
    hour_df = pd.read_parquet(s3_uri)
    dfs.append(hour_df)

df = pd.concat(dfs, ignore_index=True)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Rows: 119,708
Columns: 29


In [4]:
print(f"Number of files: {len(s3_keys)}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Number of files: 24
Rows: 119,708
Columns: 29


In [5]:
df.columns.tolist()

['datetime',
 '$schema',
 'meta',
 'id',
 'type',
 'namespace',
 'title',
 'title_url',
 'comment',
 'timestamp',
 'user',
 'bot',
 'notify_url',
 'minor',
 'length',
 'revision',
 'server_url',
 'server_name',
 'server_script_path',
 'wiki',
 'parsedcomment',
 'log_id',
 'log_type',
 'log_action',
 'log_params',
 'log_action_comment',
 'patrolled',
 'file_ts',
 'file_ts_str']

In [6]:
df.head()

,datetime,$schema,meta,id,type,namespace,title,title_url,comment,timestamp,...,wiki,parsedcomment,log_id,log_type,log_action,log_params,log_action_comment,patrolled,file_ts,file_ts_str
0,2026-09-01 04:00:02,/mediawiki/recentchange/1.0.0,{'uri': 'https://en.wikipedia.org/wiki/Jake_O%...,2.063884e+09,edit,0,Jake O'Donnell,https://en.wikipedia.org/wiki/Jake_O%27Donnell,Fix dash,1788235202,...,enwiki,Fix dash,NaN,NaN,NaN,NaN,NaN,None,2026-09-01 04:00:00,20260901_040000
1,2026-09-01 04:00:02,/mediawiki/recentchange/1.0.0,{'uri': 'https://en.wikipedia.org/wiki/Beverid...,2.063884e+09,edit,0,Beveridge Award,https://en.wikipedia.org/wiki/Beveridge_Award,Add source,1788235202,...,enwiki,Add source,NaN,NaN,NaN,NaN,NaN,None,2026-09-01 04:00:00,20260901_040000
2,2026-09-01 04:00:01,/mediawiki/recentchange/1.0.0,{'uri': 'https://en.wikipedia.org/wiki/1920_Un...,2.063884e+09,edit,0,1920 United States presidential election in Wa...,https://en.wikipedia.org/wiki/1920_United_Stat...,/* Counties that flipped from Democratic to Re...,1788235201,...,enwiki,"<span class=""autocomment""><a href=""/wiki/1920_...",NaN,NaN,NaN,NaN,NaN,None,2026-09-01 04:00:00,20260901_040000
3,2026-09-01 04:00:02,/mediawiki/recentchange/1.0.0,{'uri': 'https://en.wikipedia.org/wiki/1987_Ae...,2.063884e+09,edit,0,1987 Aegean crisis,https://en.wikipedia.org/wiki/1987_Aegean_crisis,added a bit of context,1788235202,...,enwiki,added a bit of context,NaN,NaN,NaN,NaN,NaN,None,2026-09-01 04:00:00,20260901_040000
4,2026-09-01 04:00:03,/mediawiki/recentchange/1.0.0,{'uri': 'https://en.wikipedia.org/wiki/Marvin_...,2.063884e+09,edit,0,Marvin Schwäbe,https://en.wikipedia.org/wiki/Marvin_Schw%C3%A4be,NaN,1788235203,...,enwiki,NaN,NaN,NaN,NaN,NaN,NaN,None,2026-09-01 04:00:00,20260901_040000


In [7]:
df["datetime"] = pd.to_datetime(df["datetime"])

hourly_counts = (
    df.groupby("datetime")
      .size()
      .reset_index(name="event_count")
)

hourly_counts

,datetime,event_count
0,2026-09-01 04:00:01,4
1,2026-09-01 04:00:02,3
2,2026-09-01 04:00:03,1
3,2026-09-01 04:00:04,1
4,2026-09-01 04:00:05,2
...,...,...
62599,2026-09-02 03:44:50,5
62600,2026-09-02 03:44:51,6
62601,2026-09-02 03:44:52,3
62602,2026-09-02 03:44:53,2


In [8]:
df["datetime"] = pd.to_datetime(df["datetime"])

hourly_counts = (
    df.groupby("datetime")
      .size()
      .reset_index(name="event_count")
)

hourly_counts

,datetime,event_count
0,2026-09-01 04:00:01,4
1,2026-09-01 04:00:02,3
2,2026-09-01 04:00:03,1
3,2026-09-01 04:00:04,1
4,2026-09-01 04:00:05,2
...,...,...
62599,2026-09-02 03:44:50,5
62600,2026-09-02 03:44:51,6
62601,2026-09-02 03:44:52,3
62602,2026-09-02 03:44:53,2


In [9]:
hourly_type_counts = (
    df.groupby(["file_ts", "type"])
      .size()
      .reset_index(name="event_count")
      .sort_values(["file_ts", "type"])
)

hourly_type_counts

,file_ts,type,event_count
0,2026-09-01 04:00:00,edit,3959
1,2026-09-01 04:00:00,log,440
2,2026-09-01 04:00:00,new,59
3,2026-09-01 05:00:00,edit,3429
4,2026-09-01 05:00:00,log,199
...,...,...,...
67,2026-09-02 02:00:00,log,91
68,2026-09-02 02:00:00,new,185
69,2026-09-02 03:00:00,edit,3732
70,2026-09-02 03:00:00,log,49


In [10]:
print(len(hourly_counts))
print(hourly_type_counts["type"].unique())

62604
<ArrowStringArray>
['edit', 'log', 'new']
Length: 3, dtype: str


In [11]:
from pathlib import Path

Path("images").mkdir(exist_ok=True)

hourly_counts["hour"] = pd.to_datetime(hourly_counts["file_ts"])
hourly_type_counts["hour"] = pd.to_datetime(hourly_type_counts["file_ts"])

print(hourly_counts.head())
print(hourly_type_counts.head())

KeyError: 'file_ts'

In [12]:
print(hourly_counts.columns.tolist())
print(hourly_type_counts.columns.tolist())

['datetime', 'event_count']
['file_ts', 'type', 'event_count']


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

sns.lineplot(
    data=hourly_counts,
    x="datetime",
    y="event_count",
    marker="o"
)

plt.xlabel("Hour")
plt.ylabel("Event Count")
plt.title("Wikipedia Events per Hour")
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig("images/hourly-events.png", dpi=300, bbox_inches="tight")
plt.savefig("images/hourly-events.svg", bbox_inches="tight")

plt.show()